# Validate LinkedIn URLs

### Inputs: 
- `"../derived/linkedin_company_urls.csv"`


### Outputs:
- `"../derived/linkedin_company_urls_ai_validated.csv"`
  

### Purpose:

1. Validate the retrieved URLs using AI.

Some of the retrieved URLs might be bad matches, because even if a company does not have a LinkedIn profile, a Bing search would still deliver a result. To identify these false positives, we use an AI agent that can do the following:

1. Make an assessment based on the URL if it is a likely false positive
2. If likely FP, retrieve the company profile from the URL
3. Check the company profile against the name of the company and make final judgment

In [ ]:
from langchain_dartmouth.llms import ChatDartmouthCloud

from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())

llm = ChatDartmouthCloud(model_name="google_genai.gemini-2.0-flash-001")

In [ ]:
from bs4 import BeautifulSoup
from langchain_core.messages import HumanMessage
from langchain_core.output_parsers import JsonOutputParser
from langgraph.graph import StateGraph, END
import pandas as pd
from pydantic import BaseModel
import requests

from typing import Dict, Any, Optional


class AgentState(BaseModel):
    company_name: str
    linkedin_url: str
    url_assessment: Optional[Dict[str, Any]] = None
    url_content: Optional[str] = None
    final_assessment: Optional[Dict[str, Any]] = None


def fetch_url_content(url: str) -> str:
    """Fetch the content from a URL."""
    try:
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
        }
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")

        title = soup.title.string if soup.title else ""

        description = ""
        meta_desc = soup.find("meta", {"name": "description"})
        if meta_desc:
            description = meta_desc.get("content", "")

        main_content = soup.find("main")
        main_text = (
            main_content.get_text(separator=" ", strip=True) if main_content else ""
        )

        content = f"Title: {title}\nDescription: {description}\nContent Excerpt: {main_text[:500]}..."
        return content

    except Exception as e:
        return f"Error fetching URL: {str(e)}"


# Node 1: Initial URL Assessment
def assess_url(state: AgentState) -> Dict[str, Any]:
    """Assess if the LinkedIn URL matches the company name."""
    company_name = state.company_name
    linkedin_url = state.linkedin_url

    prompt = f"""
    You are tasked with determining if a LinkedIn company URL matches the provided company name.

    Company Name: {company_name}
    LinkedIn URL: {linkedin_url}

    Based on the URL structure and company name, make an initial assessment if the URL seems correct.

    Consider the following:
    1. Does the URL contain the company name or a reasonable abbreviation/variation?
    2. Is the URL structure consistent with LinkedIn company pages (usually linkedin.com/company/...)?
    3. Are there any obvious mismatches or red flags?

    Provide your reasoning and assessment. The assessment should be exactly the term OK or SUSPICIOUS.
    Respond in JSON format with the keys 'reasoning' and 'assessment'.
    """

    chain = llm | JsonOutputParser()
    response = chain.invoke([HumanMessage(content=prompt)])

    return {"url_assessment": response}


# Router function to determine next step
def router(state: AgentState) -> str:
    """Route to the next node based on the assessment."""
    assessment = state.url_assessment["assessment"]

    if assessment != "OK":
        return "retrieve_url_content"

    return "make_final_assessment"


# Node 3: Fetch URL Content
def retrieve_url_content(state: AgentState) -> Dict[str, Any]:
    """Fetch the content from the LinkedIn URL."""
    content = fetch_url_content(state.linkedin_url)
    return {"url_content": content}


# Node 4: Make Final Assessment
def make_final_assessment(state: AgentState) -> Dict[str, Any]:
    """Make a final assessment based on all available information."""
    company_name = state.company_name
    linkedin_url = state.linkedin_url
    url_assessment = state.url_assessment
    url_content = state.url_content if state.url_content else "No content retrieved"

    prompt = f"""
    You are tasked with making a final assessment about whether a LinkedIn URL correctly matches a company.

    Company Name: {company_name}
    LinkedIn URL: {linkedin_url}

    Initial URL Assessment: {url_assessment}

    {"URL Content: " + url_content if state.url_content else "No content was retrieved as the initial assessment was confident."}

    Based on all this information, make a final determination whether the LinkedIn URL is correct for the given company.

    Provide your reasoning and a clear final verdict (CORRECT or INCORRECT).
    """

    response = llm.invoke([HumanMessage(content=prompt)])

    # Extract the final verdict
    content = response.content
    verdict = "CORRECT"
    if "INCORRECT" in content:
        verdict = "INCORRECT"

    # Create the final JSON result
    result = {
        "company_name": company_name,
        "linkedin_url": linkedin_url,
        "reasoning": content,
        "is_correct": verdict == "CORRECT",
    }

    return {"final_assessment": result}


# Build the graph
def build_graph() -> StateGraph:
    workflow = StateGraph(AgentState)

    # Add nodes
    workflow.add_node("assess_url", assess_url)
    workflow.add_node("retrieve_url_content", retrieve_url_content)
    workflow.add_node("make_final_assessment", make_final_assessment)

    # Add edges
    workflow.add_conditional_edges(
        "assess_url",
        router,
        {
            "retrieve_url_content": "retrieve_url_content",
            "make_final_assessment": "make_final_assessment",
        },
    )
    workflow.add_edge("retrieve_url_content", "make_final_assessment")
    workflow.add_edge("make_final_assessment", END)

    # Set the entry point
    workflow.set_entry_point("assess_url")

    return workflow


# Create the agent
linkedin_validator = build_graph().compile()

print(linkedin_validator.get_graph().draw_ascii())

In [ ]:
def validate_linkedin_url(company_name: str, linkedin_url: str) -> Dict[str, Any]:
    """Validate if a LinkedIn URL matches a company name."""
    initial_state = AgentState(company_name=company_name, linkedin_url=linkedin_url)

    # Run the graph
    final_state = linkedin_validator.invoke(initial_state)

    # Return the final assessment as JSON
    return final_state

In [ ]:
company_urls = pd.read_csv(
    "../data/derived/linkedin_company_urls.csv", na_values=["Not found"]
)
company_urls = company_urls.dropna(subset="Company Name")
company_urls["Reasoning"] = None
company_urls["Is Valid"] = None
company_urls

In [ ]:
def validate_url(row):
    if pd.isna(row["LinkedIn URL"]):
        return row
    if not pd.isna(row["Is Valid"]):
        return row

    result = validate_linkedin_url(
        company_name=row["Company Name"], linkedin_url=row["LinkedIn URL"]
    )
    row["Reasoning"] = result["final_assessment"]["reasoning"]
    row["Is Valid"] = result["final_assessment"]["is_correct"]
    print(row)
    return row

In [ ]:
# Read existing file, if possible
try:
    df = pd.read_csv("../data/derived/linkedin_company_urls_ai_validated.csv")
    company_urls = company_urls[["Company Name", "LinkedIn URL"]].merge(df, how="left")
except FileNotFoundError:
    print("No existing file found.")
    pass

In [ ]:
try:
    company_urls = company_urls.apply(validate_url, axis="columns")
finally:
    company_urls.to_csv(
        "../data/derived/linkedin_company_urls_ai_validated.csv", index=False
    )